# Part B - Low-Rank Adaptation (LoRA)
## TOTAL MARKS: 50
## Overview
### `Name:`
### `Roll Number:`

References:
https://magazine.sebastianraschka.com/p/lora-and-dora-from-scratch


## Instructions

<table style="width:100%; table-layout:fixed;">
<tr>

<td style="vertical-align:top; padding:12px; width:33%;">
  <h3 style="margin:0 0 8px 0; font-weight:800; color:#e53935;">
    Plagiarism Policy
  </h3>
  <ul style="margin:6px 0 0 18px; padding:0; color:#e53935;">
    <li>
      <strong>All work must be done independently.</strong><br>
      Any plagiarism or cheating (from peers or the internet) will be immediately referred to the DC.
      If you are unsure what constitutes plagiarism, consult the TAs in a timely manner.
    </li>
    <li style="margin-top:8px;">
      <strong>Do not look at anyone else’s code.</strong>
    </li>
  </ul>
</td>

<td style="vertical-align:top; padding:12px; width:33%;">
  <h3 style="margin:0 0 8px 0; font-weight:800; color:#e53935;">
    Submission Instructions
  </h3>
  <ul style="margin:6px 0 0 18px; padding:0; color:#e53935;">
    <li>
      Submit <strong>all</strong> required <code>.ipynb</code> and <code>.py</code> files on LMS
      (search how to extract scripts if confused).
    </li>
    <li style="margin-top:8px;">
      <strong>Submissions via Dropbox or email will not be accepted.</strong>
    </li>
    <li style="margin-top:8px;">
      Zip your files as <code>RollNumber_PAx.zip</code> and ensure your roll number is filled in correctly.
    </li>
    <li style="margin-top:8px;">
      <strong>Deviation from the naming convention will result in a penalty.</strong>
    </li>
    <li style="margin-top:8px;">
      <strong>Expected file structure:</strong>
    </li>
  </ul>
  <pre style="margin:8px 0 0 0; font-size:90%; color:#e53935;">
26100076_PA3.zip
├─ 26100076_PA3.ipynb
└─ 26100076_PA3.py
  </pre>
</td>

<td style="vertical-align:top; padding:12px; width:33%;">
  <h3 style="margin:0 0 8px 0; font-weight:800; color:#e53935;">
    General Instructions
  </h3>
  <ul style="margin:6px 0 0 18px; padding:0; color:#e53935;">
    <li>
      <strong>Ensure all cells are executed before submission.</strong>
    </li>
    <li style="margin-top:8px;">
      <strong>Do not remove or modify any pre-written code.</strong>
    </li>
    <li style="margin-top:8px;">
      <strong>All parts of the assignment must be attempted.</strong>
    </li>
  </ul>
</td>

</tr>
</table>

# Low-Rank Adaptation (LoRA) — An Intuitive Explanation

Large language models (LLMs) and vision transformers contain **millions or billions of parameters** (weights).  
Fine-tuning these models normally requires updating *all* of these weights, which is computationally expensive and often limited by GPU memory.

---

### Regular Fine-Tuning

In standard training or fine-tuning:

- Each layer has a large weight matrix **W**
- Training learns a full update matrix **ΔW**
- The updated weights are:

$$
W_{\text{updated}} = W + \Delta W
$$

**Drawback:**  
The update matrix **ΔW** is very large, making training slow and memory-intensive.

---

### LoRA: The Key Idea

LoRA (Low-Rank Adaptation) is based on the observation that:

> Most useful weight updates can be represented by **simple patterns**, rather than a full, complex matrix.

Instead of learning the full **ΔW**, LoRA **approximates it** using two much smaller matrices:

$$
\Delta W \approx A \cdot B
$$

where:
- **A** and **B** are small matrices
- Their matrix product has the same shape as **W**

The weight update becomes:

$$
W_{\text{updated}} = W + A \cdot B
$$

The figure below illustrates these formulas for full finetuning and LoRA side by side.

![LoRA illustration](LoRAImage.webp)

---

### Intuitive Explanation

Think of the model as a large machine with many adjustment knobs.

- **Full fine-tuning:**  
  Adjust every knob individually.

- **LoRA:**  
  Keep the original knobs fixed and add a small attachment that adjusts many knobs together in a coordinated way.

This attachment (matrices **A** and **B**) captures the most important changes while using far fewer parameters.

## Implementing a LoRA Layer (2.5 MARKS)
Implement a LoRA layer that adapts a neural network without directly modifying its original weights. Instead of learning a full weight update, represent the adaptation as a low-rank decomposition that captures task-specific changes using a compact set of parameters. This formulation reduces both memory usage and computational cost compared to full fine-tuning.

Design the LoRA layer using two trainable low-rank matrices whose product defines the weight update. Scale this update appropriately and optionally apply dropout to regulate its effect. During the forward pass, compute the low-rank update independently so it can later be combined with the output of a frozen base layer in a modular and efficient manner.

In [10]:
import torch
import torch.nn as nn
import torch.nn.functional as F


In [11]:
class LoRALayer(nn.Module):
    """
    Implements a LoRA low-rank adaptation layer.
    """

    def __init__(self, in_features, out_features, rank=8, alpha=16, dropout=0.0):
        super().__init__()

        self.rank = rank
        self.alpha = alpha
        self.scaling = alpha / rank

        self.dropout = nn.Dropout(dropout)

        # A maps input features to low-rank dimension
        self.lora_A = nn.Linear(in_features, rank, bias=False)

        # B maps low-rank dimension to output features
        self.lora_B = nn.Linear(rank, out_features, bias=False)

        # Initialize A normally
        nn.init.kaiming_uniform_(self.lora_A.weight, a=5 ** 0.5)

        # Initialize B as zero so LoRA initially changes nothing
        nn.init.zeros_(self.lora_B.weight)

    def forward(self, x):
        x = self.dropout(x)
        update = self.lora_B(self.lora_A(x))
        return update * self.scaling

## LoRA-Adapted Linear Layer (non-merged) (5 MARKS)
Implement a LoRA-adapted version of a linear layer by wrapping an existing nn.Linear module. The original weights and bias should be frozen to ensure that only the low-rank adaptation is trained. The forward pass should combine the output of the frozen linear layer with the output of the LoRA layer, illustrating how task-specific updates can be added without modifying the base parameters.

In [12]:
class LoRALinear(nn.Module):
    """
    A linear layer with LoRA adaptation.
    """

    def __init__(self, linear_layer, rank=8, alpha=16, dropout=0.0):
        super().__init__()

        # Store the original linear layer
        self.linear_layer = linear_layer

        # Freeze base weights and bias
        for param in self.linear_layer.parameters():
            param.requires_grad = False

        # Create LoRA adaptation layer
        self.lora = LoRALayer(
            in_features=linear_layer.in_features,
            out_features=linear_layer.out_features,
            rank=rank,
            alpha=alpha,
            dropout=dropout
        )

    def forward(self, x):
        # Compute output from the frozen base linear layer
        base_output = self.linear_layer(x)

        # Compute LoRA low-rank adaptation output
        lora_output = self.lora(x)

        # Combine frozen base output with trainable LoRA update
        return base_output + lora_output

## Define a Small Network with One Linear Layer (2.5 MARKS)
Create a simple experimental setup using a single linear layer and a randomly generated input tensor. Fix the random seed to ensure reproducibility. Compute and record the output of the original linear layer, which will serve as a baseline for comparison when LoRA is introduced.

In [13]:
# Set random seed for reproducibility
random_seed = 42
torch.manual_seed(random_seed)

# Create a simple linear layer
# Input size = 4, output size = 3
layer = nn.Linear(in_features=4, out_features=3)

# Create a random input tensor
# Batch size = 2, input features = 4
x = torch.randn(2, 4)

# Compute baseline output from the original linear layer
baseline_output = layer(x)

# Print baseline output
print("Input tensor:")
print(x)

print("\nBaseline output:")
print(baseline_output)

Input tensor:
tensor([[-1.2284,  0.5294,  1.2211,  0.1511],
        [-0.3319, -0.4785, -0.2631, -0.1786]])

Baseline output:
tensor([[ 0.0459,  0.0028,  0.0503],
        [-0.0072,  0.0674,  0.1393]], grad_fn=<AddmmBackward0>)


## Apply LoRA to the Linear Layer (2.5 MARKS)
Apply the LoRA-adapted linear wrapper to the previously defined layer. Run the same input through this modified layer and observe the change in output compared to the baseline. This step demonstrates how LoRA introduces an additional learned transformation while keeping the original layer weights unchanged.

In [14]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [15]:
# Wrap the original linear layer with LoRA
layer_lora = LoRALinear(
    linear_layer=layer,
    rank=2,
    alpha=4,
    dropout=0.0
)

# Compute output using the LoRA-adapted layer
lora_output = layer_lora(x)

# Print baseline and LoRA output
print("Baseline output:")
print(baseline_output)

print("\nLoRA output:")
print(lora_output)

# Print the difference between LoRA output and baseline output
print("\nDifference between LoRA output and baseline output:")
print(lora_output - baseline_output)

Baseline output:
tensor([[ 0.0459,  0.0028,  0.0503],
        [-0.0072,  0.0674,  0.1393]], grad_fn=<AddmmBackward0>)

LoRA output:
tensor([[ 0.0459,  0.0028,  0.0503],
        [-0.0072,  0.0674,  0.1393]], grad_fn=<AddBackward0>)

Difference between LoRA output and baseline output:
tensor([[0., 0., 0.],
        [0., 0., 0.]], grad_fn=<SubBackward0>)


## Merged LoRA Version (for inference) (5 MARKS)
Implement an alternative version of the linear layer in which the LoRA weight update is merged directly into the original weight matrix. In this formulation, compute the low-rank weight update and add it to the frozen base weights before applying the linear operation. This merged approach reflects how LoRA can be deployed efficiently during inference.

In [17]:
class LinearWithLoRAMerged(nn.Module):
    """
    Linear layer where LoRA weights are merged into W.
    """

    def __init__(self, linear, rank, alpha):
        super().__init__()

        # Store the original linear layer
        self.linear = linear

        # Freeze original weights and bias
        for param in self.linear.parameters():
            param.requires_grad = False

        # Create LoRA layer
        self.lora = LoRALayer(
            in_features=linear.in_features,
            out_features=linear.out_features,
            rank=rank,
            alpha=alpha,
            dropout=0.0
        )

    def forward(self, x):
        # Compute delta W = B @ A
        delta_W = self.lora.lora_B.weight @ self.lora.lora_A.weight

        # Apply LoRA scaling
        delta_W = delta_W * self.lora.scaling

        # Merge LoRA update into frozen base weight
        merged_weight = self.linear.weight + delta_W

        # Apply linear operation using merged weight
        output = F.linear(x, merged_weight, self.linear.bias)

        return output

## Verify Mathematical Equivalence  (0.5 MARKS)
Verify that the merged and non-merged LoRA implementations produce identical outputs when initialized with the same parameters. Ensure both models share the same low-rank matrices, then compare their outputs on the same input. Report the maximum absolute difference to confirm numerical equivalence between the two approaches.

In [18]:
import copy

# Create non-merged LoRA model
layer_lora_unmerged = LoRALinear(
    linear_layer=copy.deepcopy(layer),
    rank=2,
    alpha=4,
    dropout=0.0
)

# Create merged LoRA model
layer_lora_merged = LinearWithLoRAMerged(
    linear=copy.deepcopy(layer),
    rank=2,
    alpha=4
)

# Copy base linear weights and bias from unmerged to merged
layer_lora_merged.linear.weight.data.copy_(
    layer_lora_unmerged.linear_layer.weight.data
)

layer_lora_merged.linear.bias.data.copy_(
    layer_lora_unmerged.linear_layer.bias.data
)

# Copy LoRA A and B matrices from unmerged to merged
layer_lora_merged.lora.lora_A.weight.data.copy_(
    layer_lora_unmerged.lora.lora_A.weight.data
)

layer_lora_merged.lora.lora_B.weight.data.copy_(
    layer_lora_unmerged.lora.lora_B.weight.data
)

# Put both models in evaluation mode
layer_lora_unmerged.eval()
layer_lora_merged.eval()

# Compute outputs
output_unmerged = layer_lora_unmerged(x)
output_merged = layer_lora_merged(x)

# Compare outputs
max_abs_difference = torch.max(torch.abs(output_unmerged - output_merged))

print("Unmerged LoRA output:")
print(output_unmerged)

print("\nMerged LoRA output:")
print(output_merged)

print("\nMaximum absolute difference:")
print(max_abs_difference.item())

Unmerged LoRA output:
tensor([[ 0.0459,  0.0028,  0.0503],
        [-0.0072,  0.0674,  0.1393]], grad_fn=<AddBackward0>)

Merged LoRA output:
tensor([[ 0.0459,  0.0028,  0.0503],
        [-0.0072,  0.0674,  0.1393]], grad_fn=<AddmmBackward0>)

Maximum absolute difference:
0.0


The merged and non-merged LoRA implementations produce identical outputs. In the non-merged version, the output is computed as the frozen base layer output plus the LoRA update. In the merged version, the LoRA weight update is first added directly to the frozen base weight matrix, and then the linear operation is applied. Since both methods use the same low-rank matrices and scaling factor, they are mathematically equivalent. The maximum absolute difference is 0.0, confirming numerical equivalence.

# Applying LoRA Layers to LLM (2.5 MARKS)


## Preparing the Dataset
Prepare a text dataset suitable for causal language modeling. Load a subset of the AG News dataset, shuffle it for reproducibility, and split it into training and evaluation sets. Tokenize the text with fixed-length padding and truncation, and construct labels that mask padding tokens so they do not contribute to the training loss. Finally, format the dataset for PyTorch and create data loaders for batch-based training and evaluation.

In [20]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset


In [21]:


def prepare_dataset(tokenizer, max_samples=500, max_length=128):
    """
    Prepares train and test DataLoaders for language model fine-tuning.

    Args:
        tokenizer: HuggingFace tokenizer
        max_samples: number of samples to use
        max_length: maximum sequence length

    Returns:
        train_loader, test_loader
    """

    # Batch size for DataLoader
    batch_size = 8

    # Some causal LM tokenizers, such as GPT-2, do not have a pad token
    # In that case, use the EOS token as the padding token
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # Load AG News dataset
    dataset = load_dataset("ag_news", split="train")

    # Shuffle for reproducibility
    dataset = dataset.shuffle(seed=42)

    # Select a smaller subset
    dataset = dataset.select(range(max_samples))

    # Split into train and test sets
    dataset = dataset.train_test_split(test_size=0.2, seed=42)

    train_dataset = dataset["train"]
    test_dataset = dataset["test"]

    # Tokenization function
    def tokenize_function(batch):
        tokenized = tokenizer(
            batch["text"],
            padding="max_length",
            truncation=True,
            max_length=max_length
        )

        # For causal language modeling, labels are the same as input_ids
        labels = []

        for input_ids, attention_mask in zip(
            tokenized["input_ids"],
            tokenized["attention_mask"]
        ):
            label = input_ids.copy()

            # Mask padding tokens with -100 so they are ignored in loss
            label = [
                token_id if mask == 1 else -100
                for token_id, mask in zip(label, attention_mask)
            ]

            labels.append(label)

        tokenized["labels"] = labels

        return tokenized

    # Apply tokenization
    train_dataset = train_dataset.map(
        tokenize_function,
        batched=True,
        remove_columns=train_dataset.column_names
    )

    test_dataset = test_dataset.map(
        tokenize_function,
        batched=True,
        remove_columns=test_dataset.column_names
    )

    # Convert to PyTorch tensors
    train_dataset.set_format(
        type="torch",
        columns=["input_ids", "attention_mask", "labels"]
    )

    test_dataset.set_format(
        type="torch",
        columns=["input_ids", "attention_mask", "labels"]
    )

    # Create DataLoaders
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False
    )

    return train_loader, test_loader

## Initializing the Base Language Model
Load a pre-trained causal language model and its corresponding tokenizer. Configure the tokenizer to use a valid padding token and move the model to the appropriate computation device. Keep this base model in evaluation mode to serve as a frozen reference for comparison with the LoRA-adapted version.

In [23]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [24]:


# Model name
model_name = "EleutherAI/gpt-neo-125M"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# GPT-style models often do not have a padding token
# Use EOS token as padding token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load base causal language model
base_model = AutoModelForCausalLM.from_pretrained(model_name)

# Set pad token id in model config
base_model.config.pad_token_id = tokenizer.pad_token_id

# Move model to device
base_model = base_model.to(DEVICE)

# Freeze base model parameters
for param in base_model.parameters():
    param.requires_grad = False

# Set model to evaluation mode
base_model.eval()

model.safetensors:   0%|          | 0.00/526M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/160 [00:00<?, ?it/s]

GPTNeoForCausalLM LOAD REPORT from: EleutherAI/gpt-neo-125M
Key                                                   | Status     |  | 
------------------------------------------------------+------------+--+-
transformer.h.{0...11}.attn.attention.masked_bias     | UNEXPECTED |  | 
transformer.h.{0, 2, 4, 6, 8, 10}.attn.attention.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/119 [00:00<?, ?B/s]

GPTNeoForCausalLM(
  (transformer): GPTNeoModel(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(2048, 768)
    (drop): Dropout(p=0.0, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPTNeoBlock(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPTNeoAttention(
          (attention): GPTNeoSelfAttention(
            (attn_dropout): Dropout(p=0.0, inplace=False)
            (resid_dropout): Dropout(p=0.0, inplace=False)
            (k_proj): Linear(in_features=768, out_features=768, bias=False)
            (v_proj): Linear(in_features=768, out_features=768, bias=False)
            (q_proj): Linear(in_features=768, out_features=768, bias=False)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPTNeoMLP(
          (c_fc): Linear(in_features=768, out_features=3072, bias=True)
          (c_proj): Linear(in_fe

## Creating a LoRA-Adapted Model (2.5 MARKS)
Instantiate a second copy of the pre-trained language model that will be modified using LoRA. This model will be used for training and should be set to training mode. Maintaining separate base and LoRA-adapted models allows direct comparison of parameter counts and downstream performance.

In [25]:
# Load LoRA model
model_lora = AutoModelForCausalLM.from_pretrained(model_name)

# Set padding token id
model_lora.config.pad_token_id = tokenizer.pad_token_id

# Move model to device
model_lora = model_lora.to(DEVICE)

# Set training mode
model_lora.train()

Loading weights:   0%|          | 0/160 [00:00<?, ?it/s]

GPTNeoForCausalLM LOAD REPORT from: EleutherAI/gpt-neo-125M
Key                                                   | Status     |  | 
------------------------------------------------------+------------+--+-
transformer.h.{0...11}.attn.attention.masked_bias     | UNEXPECTED |  | 
transformer.h.{0, 2, 4, 6, 8, 10}.attn.attention.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


GPTNeoForCausalLM(
  (transformer): GPTNeoModel(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(2048, 768)
    (drop): Dropout(p=0.0, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPTNeoBlock(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPTNeoAttention(
          (attention): GPTNeoSelfAttention(
            (attn_dropout): Dropout(p=0.0, inplace=False)
            (resid_dropout): Dropout(p=0.0, inplace=False)
            (k_proj): Linear(in_features=768, out_features=768, bias=False)
            (v_proj): Linear(in_features=768, out_features=768, bias=False)
            (q_proj): Linear(in_features=768, out_features=768, bias=False)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPTNeoMLP(
          (c_fc): Linear(in_features=768, out_features=3072, bias=True)
          (c_proj): Linear(in_fe

## Applying LoRA to Attention Layers (2.5 MARKS)
Modify the language model by replacing selected linear layers within the attention mechanism with LoRA-adapted linear layers. Target projection layers commonly used in self-attention, such as query, key, value, and output projections. This step injects low-rank adaptations into the model while leaving the original weights structurally intact.

In [27]:
def apply_lora_to_model(model, rank=8, alpha=16, dropout=0.0):
    """
    Replaces selected attention projection Linear layers with LoRALinear wrappers.

    Args:
        model: a transformer model
        rank: LoRA rank r
        alpha: LoRA scaling alpha
        dropout: LoRA dropout

    Returns:
        model: modified model
    """

    # Freeze all original model parameters first
    for param in model.parameters():
        param.requires_grad = False

    # Attention projection names used in GPT-Neo
    target_modules = ["q_proj", "k_proj", "v_proj", "out_proj"]

    # Collect layers to replace
    modules_to_replace = []

    for module_name, module in model.named_modules():
        if isinstance(module, nn.Linear):
            last_name = module_name.split(".")[-1]

            if last_name in target_modules:
                modules_to_replace.append((module_name, module))

    # Replace selected Linear layers with LoRALinear
    for module_name, module in modules_to_replace:
        parent_name = ".".join(module_name.split(".")[:-1])
        child_name = module_name.split(".")[-1]

        parent_module = model.get_submodule(parent_name)

        setattr(
            parent_module,
            child_name,
            LoRALinear(
                linear_layer=module,
                rank=rank,
                alpha=alpha,
                dropout=dropout
            )
        )

    print(f"Replaced {len(modules_to_replace)} attention linear layers with LoRA.")

    return model

In [28]:
apply_lora_to_model(model_lora, rank=8, alpha=16)


Replaced 48 attention linear layers with LoRA.


GPTNeoForCausalLM(
  (transformer): GPTNeoModel(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(2048, 768)
    (drop): Dropout(p=0.0, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPTNeoBlock(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPTNeoAttention(
          (attention): GPTNeoSelfAttention(
            (attn_dropout): Dropout(p=0.0, inplace=False)
            (resid_dropout): Dropout(p=0.0, inplace=False)
            (k_proj): LoRALinear(
              (linear_layer): Linear(in_features=768, out_features=768, bias=False)
              (lora): LoRALayer(
                (dropout): Dropout(p=0.0, inplace=False)
                (lora_A): Linear(in_features=768, out_features=8, bias=False)
                (lora_B): Linear(in_features=8, out_features=768, bias=False)
              )
            )
            (v_proj): LoRALinear(
              (linear_layer): Linear(in_features=768, out_features=768, bias=False)
              (

## Freezing Base Parameters and Enabling LoRA Training (2.5 MARKS)
Freeze all original model parameters to prevent them from being updated during training. Enable gradient updates only for the LoRA low-rank matrices. This ensures that learning is restricted to the parameter-efficient LoRA components while the pre-trained knowledge of the base model is preserved.

In [29]:
# Freeze all parameters first
for name, param in model_lora.named_parameters():
    param.requires_grad = False

# Enable only LoRA parameters
for name, param in model_lora.named_parameters():
    if "lora_A" in name or "lora_B" in name:
        param.requires_grad = True

In [30]:
trainable_params = 0
total_params = 0

for name, param in model_lora.named_parameters():
    total_params += param.numel()

    if param.requires_grad:
        trainable_params += param.numel()
        print(name, param.shape)

print("\nTotal parameters:", total_params)
print("Trainable parameters:", trainable_params)
print("Trainable percentage:", 100 * trainable_params / total_params)

transformer.h.0.attn.attention.k_proj.lora.lora_A.weight torch.Size([8, 768])
transformer.h.0.attn.attention.k_proj.lora.lora_B.weight torch.Size([768, 8])
transformer.h.0.attn.attention.v_proj.lora.lora_A.weight torch.Size([8, 768])
transformer.h.0.attn.attention.v_proj.lora.lora_B.weight torch.Size([768, 8])
transformer.h.0.attn.attention.q_proj.lora.lora_A.weight torch.Size([8, 768])
transformer.h.0.attn.attention.q_proj.lora.lora_B.weight torch.Size([768, 8])
transformer.h.0.attn.attention.out_proj.lora.lora_A.weight torch.Size([8, 768])
transformer.h.0.attn.attention.out_proj.lora.lora_B.weight torch.Size([768, 8])
transformer.h.1.attn.attention.k_proj.lora.lora_A.weight torch.Size([8, 768])
transformer.h.1.attn.attention.k_proj.lora.lora_B.weight torch.Size([768, 8])
transformer.h.1.attn.attention.v_proj.lora.lora_A.weight torch.Size([8, 768])
transformer.h.1.attn.attention.v_proj.lora.lora_B.weight torch.Size([768, 8])
transformer.h.1.attn.attention.q_proj.lora.lora_A.weight tor

## Inspecting Trainable Parameters
Verify that only the intended LoRA parameters are trainable by inspecting the model’s parameter list. This step helps confirm that the parameter-freezing strategy has been applied correctly before training begins.

In [31]:
def check_trainable_params(model):
    """
    Prints all trainable parameters in the model and reports
    the total number of trainable parameters.
    """

    trainable_params = 0
    total_params = 0

    print("Trainable parameters:\n")

    for name, param in model.named_parameters():
        total_params += param.numel()

        if param.requires_grad:
            trainable_params += param.numel()
            print(name, param.shape)

    print("\nTotal parameters:", total_params)
    print("Trainable parameters:", trainable_params)
    print("Trainable percentage:", 100 * trainable_params / total_params)


check_trainable_params(model_lora)

Trainable parameters:

transformer.h.0.attn.attention.k_proj.lora.lora_A.weight torch.Size([8, 768])
transformer.h.0.attn.attention.k_proj.lora.lora_B.weight torch.Size([768, 8])
transformer.h.0.attn.attention.v_proj.lora.lora_A.weight torch.Size([8, 768])
transformer.h.0.attn.attention.v_proj.lora.lora_B.weight torch.Size([768, 8])
transformer.h.0.attn.attention.q_proj.lora.lora_A.weight torch.Size([8, 768])
transformer.h.0.attn.attention.q_proj.lora.lora_B.weight torch.Size([768, 8])
transformer.h.0.attn.attention.out_proj.lora.lora_A.weight torch.Size([8, 768])
transformer.h.0.attn.attention.out_proj.lora.lora_B.weight torch.Size([768, 8])
transformer.h.1.attn.attention.k_proj.lora.lora_A.weight torch.Size([8, 768])
transformer.h.1.attn.attention.k_proj.lora.lora_B.weight torch.Size([768, 8])
transformer.h.1.attn.attention.v_proj.lora.lora_A.weight torch.Size([8, 768])
transformer.h.1.attn.attention.v_proj.lora.lora_B.weight torch.Size([768, 8])
transformer.h.1.attn.attention.q_proj

## Comparing Parameter Counts  (2.5 MARKS)
Compute and compare the total number of parameters and the number of trainable parameters for both the base model and the LoRA-adapted model. Report the percentage reduction in trainable parameters achieved through LoRA to quantify its efficiency benefits.

In [32]:
def count_parameters(model):
    """
    Returns total parameters and trainable parameters.
    """
    total_params = 0
    trainable_params = 0

    for param in model.parameters():
        total_params += param.numel()

        if param.requires_grad:
            trainable_params += param.numel()

    return total_params, trainable_params

In [33]:
# Count parameters for base model
base_total_params, base_trainable_params = count_parameters(base_model)

# Count parameters for LoRA model
lora_total_params, lora_trainable_params = count_parameters(model_lora)

# For full fine-tuning, all base model parameters would be trainable
full_finetuning_trainable_params = base_total_params

# Percentage of LoRA trainable parameters compared to full fine-tuning
lora_trainable_percentage = (
    lora_trainable_params / full_finetuning_trainable_params
) * 100

# Percentage reduction in trainable parameters
reduction_percentage = (
    1 - (lora_trainable_params / full_finetuning_trainable_params)
) * 100

print("Base Model:")
print("Total parameters:", base_total_params)
print("Trainable parameters:", base_trainable_params)

print("\nLoRA-Adapted Model:")
print("Total parameters:", lora_total_params)
print("Trainable parameters:", lora_trainable_params)

print("\nComparison:")
print("Full fine-tuning trainable parameters:", full_finetuning_trainable_params)
print("LoRA trainable parameters:", lora_trainable_params)
print("LoRA trainable percentage:", lora_trainable_percentage)
print("Reduction in trainable parameters:", reduction_percentage)

Base Model:
Total parameters: 125198592
Trainable parameters: 0

LoRA-Adapted Model:
Total parameters: 125788416
Trainable parameters: 589824

Comparison:
Full fine-tuning trainable parameters: 125198592
LoRA trainable parameters: 589824
LoRA trainable percentage: 0.4711107294241776
Reduction in trainable parameters: 99.52888927057583


In [34]:
print(f"Base total parameters: {base_total_params:,}")
print(f"Base trainable parameters: {base_trainable_params:,}")

print(f"\nLoRA total parameters: {lora_total_params:,}")
print(f"LoRA trainable parameters: {lora_trainable_params:,}")

print(f"\nLoRA trains only {lora_trainable_percentage:.4f}% of full fine-tuning parameters.")
print(f"Trainable parameter reduction: {reduction_percentage:.4f}%")

Base total parameters: 125,198,592
Base trainable parameters: 0

LoRA total parameters: 125,788,416
LoRA trainable parameters: 589,824

LoRA trains only 0.4711% of full fine-tuning parameters.
Trainable parameter reduction: 99.5289%


## Training the LoRA-Adapted Model  (2.5 MARKS)
Train the LoRA-adapted model using the prepared dataset and an appropriate optimizer. Perform multiple training epochs while monitoring the training loss. Since only a small subset of parameters is being updated, this training process is significantly more efficient than full fine-tuning.

In [35]:
# Prepare dataset
train_loader, test_loader = prepare_dataset(
    tokenizer=tokenizer,
    max_samples=500,
    max_length=128
)

# Optimizer should update only trainable LoRA parameters
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model_lora.parameters()),
    lr=1e-4
)

# Set LoRA model to training mode
model_lora.train()

num_epochs = 2

for epoch in range(num_epochs):
    total_loss = 0.0

    for step, batch in enumerate(train_loader):
        # Move batch to device
        batch = {
            key: value.to(DEVICE)
            for key, value in batch.items()
        }

        # Clear old gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model_lora(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            labels=batch["labels"]
        )

        loss = outputs.loss

        # Backward pass
        loss.backward()

        # Update only LoRA parameters
        optimizer.step()

        total_loss += loss.item()

        # Print loss every few steps
        if step % 10 == 0:
            print(
                f"Epoch [{epoch + 1}/{num_epochs}], "
                f"Step [{step}/{len(train_loader)}], "
                f"Loss: {loss.item():.4f}"
            )

    avg_loss = total_loss / len(train_loader)

    print(f"Epoch [{epoch + 1}/{num_epochs}] Average Training Loss: {avg_loss:.4f}")

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Epoch [1/2], Step [0/50], Loss: 4.4227
Epoch [1/2], Step [10/50], Loss: 4.4689
Epoch [1/2], Step [20/50], Loss: 4.3005
Epoch [1/2], Step [30/50], Loss: 4.3836
Epoch [1/2], Step [40/50], Loss: 4.5421
Epoch [1/2] Average Training Loss: 4.3207
Epoch [2/2], Step [0/50], Loss: 4.0032
Epoch [2/2], Step [10/50], Loss: 4.0522
Epoch [2/2], Step [20/50], Loss: 3.7172
Epoch [2/2], Step [30/50], Loss: 4.1603
Epoch [2/2], Step [40/50], Loss: 4.5636
Epoch [2/2] Average Training Loss: 4.1576


## Evaluating Model Accuracy  (5 MARKS)
Evaluate both the original base model and the LoRA-adapted model on the test dataset. Compute token-level accuracy while ignoring masked positions. Compare the results to assess how well the LoRA-based fine-tuning performs relative to the frozen pre-trained model.

In [36]:
@torch.no_grad()
def compute_accuracy(model, dataloader, device):
    """
    Computes token-level accuracy for causal language modeling.

    Steps:
    1. Set model to eval mode
    2. Get logits for each batch
    3. Shift logits and labels for next-token prediction
    4. Compute predictions using argmax
    5. Ignore labels with value -100
    6. Return accuracy percentage
    """

    model.eval()

    correct = 0
    total = 0

    for batch in dataloader:
        # Move batch to device
        batch = {
            key: value.to(device)
            for key, value in batch.items()
        }

        # Forward pass
        outputs = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"]
        )

        logits = outputs.logits
        labels = batch["labels"]

        # For causal LM:
        # logits at position t predict token at position t+1
        shift_logits = logits[:, :-1, :]
        shift_labels = labels[:, 1:]

        # Get predicted token ids
        predictions = torch.argmax(shift_logits, dim=-1)

        # Ignore padding/masked positions
        mask = shift_labels != -100

        # Count correct predictions
        correct += (predictions[mask] == shift_labels[mask]).sum().item()
        total += mask.sum().item()

    accuracy = 100 * correct / total

    return accuracy

In [39]:

print(f"Test accuracy orig model: {compute_accuracy(base_model, test_loader, DEVICE):.2f}%")
print(f"Test accuracy LoRA model: {compute_accuracy(model_lora, test_loader, DEVICE):.2f}%")

Test accuracy orig model: 29.51%
Test accuracy LoRA model: 31.73%


# Reflective Questions (12 Marks)
Answer the reflective questions provided below.
Q1. Why is representing weight updates as a low-rank decomposition effective for large language models? \
Ans. \
Q2. What assumptions does LoRA make about the nature of task-specific adaptations?\
Ans. \
Q3. Why are LoRA layers typically applied to attention projection layers rather than all linear layers?\
Ans. \
Q4. How does the choice of rank affect the expressive capacity of the LoRA adaptation?\
Ans. \
Q5. Why might a LoRA-adapted model perform comparably to a fully fine-tuned model on some tasks?\
Ans. \
Q6. Under what conditions could LoRA lead to worse performance than full fine-tuning?
Ans.


**Q1. Why is representing weight updates as a low-rank decomposition effective for large language models?**

Large language models have millions or billions of parameters, so updating all weights during fine-tuning is expensive. LoRA is effective because it assumes that the important changes needed for a new task can be captured using much smaller matrices. Instead of learning a huge full update matrix, LoRA learns two small low-rank matrices. This greatly reduces the number of trainable parameters, memory usage, and training cost while still allowing the model to adapt to the new task

**Q2. What assumptions does LoRA make about the nature of task-specific adaptations?**

LoRA assumes that task-specific changes do not require changing every weight in the model independently. Instead, it assumes that the useful changes follow simpler patterns that can be represented in a lower-dimensional space. In other words, LoRA assumes that most of the knowledge in the pre-trained model is already useful, and only a small, structured adjustment is needed for a new task

**Q3. Why are LoRA layers typically applied to attention projection layers rather than all linear layers?**

Attention projection layers are very important in transformer models because they control how tokens attend to each other. The query, key, value, and output projection layers strongly affect how the model understands relationships between words. Applying LoRA to these layers gives the model enough flexibility to adapt while keeping the number of trainable parameters small Applying LoRA to all linear layers would increase memory and computation cost which reduces the main advantage of LoRA

**Q4. How does the choice of rank affect the expressive capacity of the LoRA adaptation?**

The rank controls how much information the LoRA update can represent. A smaller rank uses fewer parameters and is more efficient, but it may not capture enough task-specific changes. A larger rank gives the LoRA layer more expressive power, meaning it can learn more complex adaptations. However, increasing the rank also increases the number of trainable parameters and memory usage. So, rank is a trade-off between efficiency and performance.

**Q5. Why might a LoRA-adapted model perform comparably to a fully fine-tuned model on some tasks?**

A LoRA-adapted model can perform comparably to a fully fine-tuned model because the pre-trained model already contains a lot of general language knowledge. For many tasks, the model does not need to relearn everything. It only needs small adjustments to behave better for the specific dataset or task. LoRA provides these small adjustments through trainable low-rank matrices, which can be enough to achieve similar performance to full fine-tuning.

**Q6. Under what conditions could LoRA lead to worse performance than full fine-tuning?**

LoRA may perform worse when the task requires large changes to the model's behavior or knowledge. If the new task is very different from the data the model was originally trained on, small low-rank updates may not be powerful enough. LoRA can also perform worse if the rank is too small, if LoRA is applied to too few layers, or if the dataset is large and complex enough that full fine-tuning would benefit from updating more parameters. In these cases, full fine-tuning may give the model more flexibility to learn the task.

# AI Usage Report

## Assignment Title

Low-Rank Adaptation (LoRA) Implementation and Application to a Language Model

## AI Tool Used

**Tool:** ChatGPT 5.5 Thinking  
**Provider:** OpenAI  
**Year:** 2026  
**Account Type:** OpenAI Plus Account  

## Purpose of AI Use

Generative AI was used as a learning and coding support tool for this assignment. It was used to help explain the assignment requirements, structure the implementation, debug errors, and write explanatory text for the report.

The AI tool was not used as a replacement for understanding the assignment. The generated code and explanations were reviewed, executed, tested, and modified where necessary before being included.

## Areas Where AI Was Used

AI assistance was used for the following parts of the assignment:

1. Understanding the intuition behind Low-Rank Adaptation (LoRA).
2. Implementing a standalone `LoRALayer` using two low-rank matrices.
3. Implementing a non-merged `LoRALinear` wrapper around an existing `nn.Linear` layer.
4. Implementing a merged LoRA linear layer for inference.
5. Verifying mathematical equivalence between merged and non-merged LoRA implementations.
6. Preparing the AG News dataset for causal language modeling.
7. Loading GPT-Neo-125M as the base causal language model.
8. Creating a second LoRA-adapted model.
9. Applying LoRA to selected attention projection layers.
10. Freezing base model parameters and enabling only LoRA parameters for training.
11. Inspecting trainable parameters.
12. Comparing parameter counts between full fine-tuning and LoRA fine-tuning.
13. Writing the training loop for the LoRA-adapted model.
14. Writing the token-level accuracy evaluation function.
15. Explaining and interpreting experimental outputs.

## Prompts Used

The following types of prompts were used with ChatGPT:

| Prompt Area | Description |
|---|---|
| LoRA explanation | Asked ChatGPT to explain the assignment requirements and the intuition behind LoRA. |
| LoRA layer implementation | Asked for help completing the `LoRALayer` class using low-rank matrices `A` and `B`. |
| Non-merged LoRA layer | Asked for help implementing `LoRALinear`, which adds LoRA output to a frozen base linear layer. |
| Merged LoRA layer | Asked for help implementing an inference version where the LoRA update is merged into the base weight matrix. |
| Mathematical verification | Asked how to compare merged and non-merged LoRA outputs using the same parameters. |
| Dataset preparation | Asked for help preparing the AG News dataset for causal language modeling. |
| Base model setup | Asked how to load the tokenizer and GPT-Neo-125M model correctly. |
| Applying LoRA to LLM | Asked how to replace attention projection layers with LoRA-adapted layers. |
| Parameter freezing | Asked how to freeze all base model parameters and enable gradients only for LoRA matrices. |
| Parameter counting | Asked how to compare trainable parameters between full fine-tuning and LoRA fine-tuning. |
| Training loop | Asked for help writing the LoRA training loop and printing loss. |
| Evaluation | Asked for help computing token-level accuracy while ignoring masked padding positions. |
| Report writing | Asked for help writing explanations and the AI policy report. |

## AI-Generated Content Used

The AI-generated responses provided:

- Code templates for PyTorch classes and functions.
- Explanations of LoRA concepts.
- Debugging guidance for errors such as undefined classes.
- Suggested verification steps.
- Suggested wording for report sections.
- Interpretation of parameter-count results.

## Edits and Verification Performed

The generated code was reviewed and executed in Google Colab. The following checks were performed:

- Confirmed that the original linear layer weights were frozen.
- Confirmed that only `lora_A.weight` and `lora_B.weight` were trainable.
- Verified that the baseline and LoRA outputs were initially identical because `lora_B` was initialized to zero.
- Verified that merged and non-merged LoRA implementations produced identical outputs.
- Confirmed that LoRA was applied to attention projection layers such as `q_proj`, `k_proj`, `v_proj`, and `out_proj`.
- Confirmed that the LoRA-adapted model had significantly fewer trainable parameters than full fine-tuning.
- Ran the training loop and monitored loss values.
- Used masked labels with `-100` so padding tokens did not contribute to the loss or token-level accuracy.

## Parameter Count Summary

The base model contained:

- **Total parameters:** 125,198,592
- **Trainable parameters:** 0, because the base model was frozen as a reference model

The LoRA-adapted model contained:

- **Total parameters:** 125,788,416
- **Trainable parameters:** 589,824

For full fine-tuning, the number of trainable parameters would be equal to the base model size:

\[
125,198,592
\]

With LoRA, only:

\[
589,824
\]

parameters were trained.

The percentage of trainable parameters used by LoRA was:

\[
0.4711\%
\]

The reduction in trainable parameters was:

\[
99.5289\%
\]

This confirms that LoRA greatly reduced the number of parameters updated during training.

## Responsibility Statement

I understand that I am responsible for the correctness, accuracy, and originality of the submitted work. The AI-generated content was used as support for understanding, implementation, debugging, and explanation. The code was reviewed, executed, and checked before submission.

I also understand that using AI does not remove responsibility for ensuring that the submitted work meets the assignment requirements and does not match another student’s submission without proper acknowledgement.

## AI Citation

ChatGPT 5.5 Thinking (2026). Prompt: Assistance with implementing and explaining Low-Rank Adaptation layers, LoRA-adapted linear layers, merged LoRA inference, dataset preparation, LoRA application to GPT-Neo-125M, parameter counting, training, evaluation, debugging, and report writing. Response: Provided explanations, code templates, debugging support, and report-writing assistance. OpenAI Plus Account.

## chat Link: https://chatgpt.com/share/69f9bf0c-c3c8-83a4-8e66-c141824e9585
